## Example 1: Basic HTML Sanitization and XSS Prevention

In [ ]:
# Example 1: Basic HTML Sanitization and XSS Prevention
from lxml_html_clean import Cleaner, clean_html

print("=" * 70)
print(f"🚀 Example 1: Basic HTML Sanitization")
print("=" * 70)

# Dangerous HTML content with potential XSS attacks
dangerous_html = '''
<div>
    <h1>Article Title</h1>
    <p>This is safe content.</p>
    <script>alert('XSS Attack!');</script>
    <p onclick="maliciousFunction()">Click me</p>
    <img src="x" onerror="alert('Image XSS')">
    <iframe src="javascript:alert('Frame XSS')"></iframe>
</div>
'''

print("Dangerous HTML Input:")
print(dangerous_html.strip())

# Method 1: a default Cleaner
safe_html_1 = Cleaner().clean_html(dangerous_html)
print(f"\n✅ Cleaned with Cleaner():")
print(safe_html_1.strip())

# Method 2: the module-level clean_html() uses the same defaults
safe_html_2 = clean_html(dangerous_html)
assert safe_html_2 == safe_html_1

# Script elements, event-handler attributes and frames are gone; the content stays
assert "<script" not in safe_html_1 and "XSS Attack" not in safe_html_1, safe_html_1
assert "onclick" not in safe_html_1 and "onerror" not in safe_html_1, safe_html_1
assert "<iframe" not in safe_html_1 and "javascript:" not in safe_html_1, safe_html_1
for kept in ("<h1>Article Title</h1>", "<p>This is safe content.</p>", "<p>Click me</p>", '<img src="x">'):
    assert kept in safe_html_1, (kept, safe_html_1)
print("\n✅ <script>, on* handlers and the javascript: <iframe> were removed; text and structure kept")

## Example 2: Custom Cleaning Rules and Selective Sanitization

In [ ]:
# Example 2: Custom Cleaning Rules and Selective Sanitization
from lxml_html_clean import Cleaner

print("=" * 70)
print(f"🚀 Example 2: Custom Cleaning Rules")
print("=" * 70)

# HTML with mixed content - some good, some potentially dangerous
mixed_html = '''
<article>
    <h2>Blog Post</h2>
    <p style="color: blue;">Styled paragraph</p>
    <a href="https://example.com">External link</a>
    <form action="/search"><input type="text" name="q"></form>
    <script>trackPageView();</script>
    <img src="photo.jpg" alt="Photo">
    <iframe src="video.html" width="400" height="300"></iframe>
</article>
'''

print("Mixed HTML Content:")
print(mixed_html.strip())

# Cleaner's defaults already remove forms, frames, embedded content and non-safe attributes,
# so a conservative profile has to switch those off explicitly
conservative = Cleaner(scripts=True, javascript=True, style=False, forms=False,
                       embedded=False, frames=False, safe_attrs_only=False)
conservative_result = conservative.clean_html(mixed_html)
print(f"\n🟡 Conservative Cleaning (scripts only):")
print(conservative_result.strip())
assert "<script" not in conservative_result
assert '<p style="color: blue;">' in conservative_result and "<form" in conservative_result
assert "<iframe" in conservative_result

# Moderate: also styles and forms
moderate = Cleaner(scripts=True, javascript=True, style=True, forms=True,
                   embedded=False, frames=False, safe_attrs_only=False)
moderate_result = moderate.clean_html(mixed_html)
print(f"\n🟠 Moderate Cleaning (scripts, styles, forms):")
print(moderate_result.strip())
assert "<script" not in moderate_result and "<form" not in moderate_result and "<input" not in moderate_result
assert "style=" not in moderate_result and "<p>Styled paragraph</p>" in moderate_result
assert "<iframe" in moderate_result

# Strict: frames and embedded content as well
strict = Cleaner(scripts=True, javascript=True, style=True, forms=True, embedded=True, frames=True)
strict_result = strict.clean_html(mixed_html)
print(f"\n🔴 Strict Cleaning (most interactive elements):")
print(strict_result.strip())
assert "<iframe" not in strict_result and "<form" not in strict_result and "<script" not in strict_result
for kept in ("<h2>Blog Post</h2>", '<a href="https://example.com">External link</a>', '<img src="photo.jpg" alt="Photo">'):
    assert kept in strict_result, (kept, strict_result)

print("\n✅ Each profile removes exactly what it is configured to remove")

## Example 3: Advanced Text Processing with Autolink and Word Breaking

In [ ]:
# Example 3: Advanced Text Processing with Autolink and Word Breaking
from lxml_html_clean import autolink_html, word_break_html, Cleaner

print("=" * 70)
print(f"🚀 Example 3: Advanced Text Processing")
print("=" * 70)

# Text content with URLs and long words
text_content = '''
<div>
    <p>Visit our website at https://www.python.org/downloads/ for more info.</p>
    <p>Local test servers such as http://localhost:8000/ are left alone.</p>
    <p>Supercalifragilisticexpialidocious is a verylongwordthatmightbreakthelayout.</p>
</div>
'''

print("Original Text Content:")
print(text_content.strip())

# Step 1: Clean the HTML first for safety
safe_content = Cleaner().clean_html(text_content)

# Step 2: Turn bare URLs into links (localhost and example.* hosts are skipped by default)
autolinked_content = autolink_html(safe_content)
print(f"\n🔗 After Auto-linking URLs:")
print(autolinked_content.strip())
assert '<a href="https://www.python.org/downloads/">https://www.python.org/downloads/</a>' in autolinked_content
assert 'href="http://localhost:8000/"' not in autolinked_content

# Step 3: Insert zero-width spaces into words longer than max_width so they can wrap
final_content = word_break_html(autolinked_content, max_width=25)
print(f"\n📏 After Word Breaking (max_width=25):")
print(final_content.strip())
assert "Supercalifragilisticexpia\u200blidocious" in final_content, final_content
assert "Visit our website at" in final_content  # words within max_width are left alone
assert final_content.replace("\u200b", "") == autolinked_content

print("✅ Auto-linking and word breaking checked")

## Example 4: Link Processing, NoFollow and Embedded-Content Host Whitelist

In [ ]:
# Example 4: Link Processing, NoFollow and Embedded-Content Host Whitelist
from lxml_html_clean import Cleaner

print("=" * 70)
print(f"🚀 Example 4: Link Processing, NoFollow and Host Whitelist")
print("=" * 70)

# HTML with various types of links
link_html = '''
<div>
    <h3>Different Link Types</h3>
    <link rel="stylesheet" href="https://cdn.example.com/site.css">
    <a href="https://trusted-site.com">Trusted External Link</a>
    <a href="/internal-page">Internal Link</a>
    <a href="javascript:alert('xss')">Malicious JS Link</a>
    <form action="/submit" method="post">
        <input type="text" name="data">
    </form>
</div>
'''

print("Original HTML with Links:")
print(link_html.strip())

# add_nofollow=True adds rel="nofollow" to every <a>; javascript: URLs are emptied either way
safe_links = Cleaner(add_nofollow=True).clean_html(link_html)
print(f"\n🔗 Links with NoFollow Added:")
print(safe_links.strip())
assert '<a href="https://trusted-site.com" rel="nofollow">' in safe_links, safe_links
assert '<a href="/internal-page" rel="nofollow">' in safe_links, safe_links
assert "javascript:" not in safe_links, safe_links

# links=True removes <link> elements; <a> anchors are kept. forms=True removes the form
no_links = Cleaner(links=True, forms=True).clean_html(link_html)
print(f"\n🚫 <link> elements and forms removed:")
print(no_links.strip())
assert "<link" not in no_links and "<form" not in no_links and "<input" not in no_links, no_links
assert '<a href="https://trusted-site.com">Trusted External Link</a>' in no_links, no_links

# host_whitelist governs embedded content (<iframe>, <embed>, <object>), not ordinary links
embed_html = '''
<div>
    <p>Two embedded videos:</p>
    <iframe src="https://www.youtube.com/embed/abc123"></iframe>
    <iframe src="https://videos.untrusted.example/embed/xyz"></iframe>
</div>
'''
whitelist_result = Cleaner(host_whitelist=["www.youtube.com"]).clean_html(embed_html)
print(f"\n✅ Embedded content with host_whitelist=['www.youtube.com']:")
print(whitelist_result.strip())
assert '<iframe src="https://www.youtube.com/embed/abc123"></iframe>' in whitelist_result, whitelist_result
assert "untrusted.example" not in whitelist_result, whitelist_result

print("\n✅ nofollow, <link>/form removal and the embed host whitelist checked")

## Example 5: Advanced Tag Whitelisting and Attribute Control

In [ ]:
# Example 5: Advanced Tag Whitelisting and Attribute Control
print("=" * 70)
print(f"🚀 Example 5: Advanced Tag Whitelisting and Attribute Control")
print("=" * 70)

from lxml_html_clean import Cleaner

# Complex HTML with various tags and attributes
complex_html = '''
    <article data-id="123" class="post">
        <h2 id="title" onclick="track()">Article Title</h2>
        <p class="intro" style="color: red;">Introduction paragraph</p>
        <blockquote cite="source.html">Famous quote</blockquote>
        <code class="python">print("hello")</code>
        <img src="photo.jpg" alt="Photo" width="300" onload="evil()">
        <table border="1" cellpadding="5">
            <tr><td>Data</td></tr>
        </table>
        <div contenteditable="true">Editable content</div>
        <span style="font-weight: bold;">Bold text</span>
    </article>
    '''

print("Complex HTML with Various Tags:")
print(complex_html.strip())

# Whitelist-based cleaner - only allow specific tags
# `allow_tags` and `remove_unknown_tags` are MUTUALLY EXCLUSIVE and lxml_html_clean raises
# `ValueError: It does not make sense to pass in both ...`. An explicit allow-list already
# implies that anything absent from it goes, so `remove_unknown_tags` must be turned OFF.
whitelist_cleaner = Cleaner(
    allow_tags=['article', 'h2', 'p', 'blockquote', 'code', 'span'],
    remove_unknown_tags=False,  # implied by allow_tags, and rejected alongside it
    safe_attrs_only=True,      # Only allow safe attributes
    safe_attrs=['class', 'id', 'cite', 'alt']  # Allowed attributes
)

whitelisted_result = whitelist_cleaner.clean_html(complex_html)

print(f"\n✅ Whitelist Cleaned (specific tags + safe attributes):")
print(whitelisted_result.strip())

# Tag removal vs content preservation
kill_tags_cleaner = Cleaner(
    kill_tags=['img', 'table'],        # Remove these tags completely
    remove_tags=['div'],               # Remove tag but keep content
    safe_attrs_only=True
)

tag_removal_result = kill_tags_cleaner.clean_html(complex_html)

print(f"\n🗑️ With Tag Removal (kill vs remove):")
print(tag_removal_result.strip())

# Comments and processing instructions
html_with_comments = '''
    <div>
        <!-- This is a comment -->
        <p>Normal content</p>
        <!--[if IE]><p>IE only content</p><![endif]-->
        <?xml version="1.0"?>
    </div>
    '''

comments_cleaner = Cleaner(
    comments=True,                    # Remove comments
    # Cleaner has no separate switch for IE conditional comments: they are ordinary comments, so comments=True strips them.
    processing_instructions=True      # Remove processing instructions
)

comments_result = comments_cleaner.clean_html(html_with_comments)

print(f"\n💬 Comments and Processing Instructions Removal:")
print(f"Before: {html_with_comments.strip()}")
print(f"After:  {comments_result.strip()}")
assert "Normal content" in comments_result, comments_result
assert "This is a comment" not in comments_result and "IE only content" not in comments_result, comments_result

print(f"\n🎯 Advanced Cleaning Features:")
print("• Tag whitelisting for precise control")
print("• Safe attributes filtering")
print("• Kill vs remove tag behavior")
print("• Comments and conditional comments removal")

print("✅ Advanced tag control and whitelisting completed")